# Пример решения универсальной задачи "обоснованного выбора лучшего из двух вариантов" c верификацией аспектов через webcite.co



**Для запуска вычислений, в секретах (Keys) этого .ipynb ноутбука необходимо задать значения следующих переменных:**
- `OPENAI_BASE_URL` (пример: `https://api.deepseek.com`)
- `OPENAI_API_KEY` (пример: `sk-...`)
- `OPENAI_MODEL` (пример: `deepseek-chat`)
- `WEBCITE_API_KEY` (пример: `webcite...`)


## Настройка среды, вспомогательных функций и общих переменных для вычисления

Установка зависимостей и пакета [core-kbt](https://https://github.com/ady1981/core-kbt):

In [1]:
#!rm -r /content/processes ## Delete process cache

In [2]:
!pip install -q poetry
!poetry config virtualenvs.in-project false
!rm -r /content/core-kbt
%cd /content
!git clone https://github.com/ady1981/core-kbt ## -b with-webcite
%cd /content/core-kbt
!mv -n /content/core-kbt/processes /content/processes ## Keep cache of process results separately in /content/processes
!rm -r /content/core-kbt/processes
!ln -s /content/processes /content/core-kbt/processes ## Link to the cache
!pip install ruamel.yaml
!poetry install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.3/292.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 464.5/464.5 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 120.3 MB/s eta 0:00:00
rm: cannot remove '/content/core-kbt': No such file or directory
/content
Cloning into 'core-kbt'...
remote: Enumerating objects: 1920, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing 

Установка дополнительных параметров для вычисления:

In [3]:
frame_of_reference = 'Unbiased logical framework'
output_content_language = 'English'
extra_information_retrieval_strategy = 'Only_unbiased_authoritative_sources: true'

Установка переменных среды для openAI-compatible API:

In [4]:
import os
from google.colab import userdata
os.environ['OPENAI_BASE_URL'] = userdata.get('OPENAI_BASE_URL')
os.environ['OPENAI_MODEL'] = userdata.get('OPENAI_MODEL') ## Название LLM модели
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['WEBCITE_API_KEY'] = userdata.get('WEBCITE_API_KEY')

Вспомогательные функции для вычисления:

In [5]:
import asyncio
import os
import pandas as pd
from google.colab import data_table

from kbt_core.ai_function import evaluate_function
from kbt_core.common import with_model_input_data, get_float

data_table.enable_dataframe_formatter()

async def calc_concept_aspect_comparison(observer_context_description, a_concept, b_concept, frame_of_reference, output_content_language, extra_information_retrieval_strategy):
    input_data = {
        'observer_context_description': observer_context_description,
        'frame_of_reference': frame_of_reference,
        'a_concept': a_concept,
        'b_concept': b_concept,
        'output_content_language': output_content_language,
        'extra_information_retrieval_strategy': extra_information_retrieval_strategy
    }
    return await evaluate_function('webcite_concept_aspect_comparison', with_model_input_data(input_data, os.environ['OPENAI_MODEL']))

def create_output_tables(result, observer_context_description, a_concept, b_concept):
  llm_model = os.environ['OPENAI_MODEL']
  total_cols = {
      'result_name': ['comparison_winner', 'a_concept_score', 'b_concept_score', 'LLM_model', 'superordinate_concept', 'perspective_observer_strategy', 'point_of_view', 'observer_context_description', 'a_concept', 'b_concept'],
      'result_value':
        [result['comparison_winner'],
          f'{round(result["comparison_total"] * 100)}%',
          f'{round(-result["comparison_total"] * 100)}%',
         llm_model] +
          [result[c] for c in ['superordinate_concept', 'perspective_observer_strategy', 'point_of_view']] +
          [observer_context_description, a_concept, b_concept]
      }
  aspect_cols = [{'type': 'feature_score',
  'aspect': aspect['aspect_name'],
  'feature': feature['feature_name'],
  'a_concept_winner': get_float(feature, 'comparison', 0) > 0,
  'a_concept_score': f'{round(feature["normalized_comparison"] * 100)}%' if get_float(feature, 'comparison', 0) > 0 else '-',
  'b_concept_winner': get_float(feature, 'comparison', 0) < 0,
  'b_concept_score': f'{round(-feature["normalized_comparison"] * 100)}%' if get_float(feature, 'comparison', 0) < 0 else '-'}
      for aspect in result['perspective_aspects']
      for feature in aspect['aspect_features']]
  total_dt = pd.DataFrame(total_cols)
  aspect_dt = pd.DataFrame(aspect_cols)
  return (total_dt, aspect_dt)


## Пример №1: Выбор фреймворка для разработки нового микросервиса для владельца

In [9]:
d01_observer_context_description = 'Владельцу продукта необходимо выбрать оптимальный фреймворк для разработки нового микросервиса среднего размера, обеспечив его запуск в течение 60 календарных дней. Приоритетными целями являются максимальная скорость итеративной разработки.'
d01_a_concept = 'Spring Boot (Java)'
d01_b_concept = 'FastAPI (Python)'
d01_result = await calc_concept_aspect_comparison(d01_observer_context_description, d01_a_concept, d01_b_concept, frame_of_reference, output_content_language, extra_information_retrieval_strategy)
(d01_total_dt, d01_aspect_dt) = create_output_tables(d01_result, d01_observer_context_description, d01_a_concept, d01_b_concept)


start: input_id=superordinate_concept_identification_af.1.3f8914212a77e71bfe150190c5b8514c.00cdbcc4ac227e101271e729b3e737c2
process:
{"concepts": "- Spring Boot (Java)\n- FastAPI (Python)", "context_knowledge_specification": "### Context description\n\u0412\u043b\u0430\u0434\u0435\u043b\u044c\u0446\u0443 \u043f\u0440\u043e\u0434\u0443\u043a\u0442\u0430 \u043d\u0435\u043e\u0431\u0445\u043e\u0434\u0438\u043c\u043e \u0432\u044b\u0431\u0440\u0430\u0442\u044c \u043e\u043f\u0442\u0438\u043c\u0430\u043b\u044c\u043d\u044b\u0439 \u0444\u0440\u0435\u0439\u043c\u0432\u043e\u0440\u043a \u0434\u043b\u044f \u0440\u0430\u0437\u0440\u0430\u0431\u043e\u0442\u043a\u0438 \u043d\u043e\u0432\u043e\u0433\u043e \u043c\u0438\u043a\u0440\u043e\u0441\u0435\u0440\u0432\u0438\u0441\u0430 \u0441\u0440\u0435\u0434\u043d\u0435\u0433\u043e \u0440\u0430\u0437\u043c\u0435\u0440\u0430, \u043e\u0431\u0435\u0441\u043f\u0435\u0447\u0438\u0432 \u0435\u0433\u043e \u0437\u0430\u043f\u0443\u0441\u043a \u0432 \u0442\u0435\u0447

In [10]:
d01_total_dt


,result_name,result_value
0,comparison_winner,FastAPI (Python)
1,a_concept_score,-11%
2,b_concept_score,11%
3,LLM_model,deepseek/deepseek-pro
4,superordinate_concept,Web application frameworks
5,perspective_observer_strategy,Product owner
6,point_of_view,Maximizing development speed
7,observer_context_description,Владельцу продукта необходимо выбрать оптималь...
8,a_concept,Spring Boot (Java)
9,b_concept,FastAPI (Python)


In [11]:
d01_aspect_dt

,type,aspect,feature,a_concept_winner,a_concept_score,b_concept_winner,b_concept_score
0,feature_score,Development speed,Time to first feature,False,-,True,11%
1,feature_score,Development speed,Scaffolding speed,False,-,False,-
2,feature_score,Development speed,Hot reload support,False,-,False,-
3,feature_score,Ease of learning,Learning curve,False,-,False,-
4,feature_score,Ease of learning,Documentation quality,False,-,False,-
5,feature_score,Ease of learning,Community size,False,-,False,-
6,feature_score,Microservice suitability,Lightweight footprint,False,-,False,-
7,feature_score,Microservice suitability,Built-in HTTP server,False,-,False,-
8,feature_score,Microservice suitability,Containerization support,False,-,False,-
9,feature_score,Iterative development support,Rapid prototyping capability,False,-,False,-


In [12]:
#!rm /content/processes/by_status/terminated/webcite_verify.1.d16897fc25a88cbfc980bebf449aabdf.json

## Пример №2: Выбор лучшего метода хранения данных для долгосрочного архива

In [6]:
d02_observer_context_description = 'Крупная организация (архив) должна выбрать, какой формат использовать для хранения цифровых копий исторических документов на следующие 100 лет, с акцентом на долговечность и независимость от проприетарного ПО.'
d02_a_concept = 'PDF'
d02_b_concept = 'TIFF'
d02_result = await calc_concept_aspect_comparison(d02_observer_context_description, d02_a_concept, d02_b_concept, frame_of_reference, output_content_language, extra_information_retrieval_strategy)
(d02_total_dt, d02_aspect_dt) = create_output_tables(d02_result, d02_observer_context_description, d02_a_concept, d02_b_concept)


start: input_id=superordinate_concept_identification_af.1.3f8914212a77e71bfe150190c5b8514c.1bc5be3867816d67525718051160b9f5
process:
{"concepts": "- PDF\n- TIFF", "context_knowledge_specification": "### Context description\n\u041a\u0440\u0443\u043f\u043d\u0430\u044f \u043e\u0440\u0433\u0430\u043d\u0438\u0437\u0430\u0446\u0438\u044f (\u0430\u0440\u0445\u0438\u0432) \u0434\u043e\u043b\u0436\u043d\u0430 \u0432\u044b\u0431\u0440\u0430\u0442\u044c, \u043a\u0430\u043a\u043e\u0439 \u0444\u043e\u0440\u043c\u0430\u0442 \u0438\u0441\u043f\u043e\u043b\u044c\u0437\u043e\u0432\u0430\u0442\u044c \u0434\u043b\u044f \u0445\u0440\u0430\u043d\u0435\u043d\u0438\u044f \u0446\u0438\u0444\u0440\u043e\u0432\u044b\u0445 \u043a\u043e\u043f\u0438\u0439 \u0438\u0441\u0442\u043e\u0440\u0438\u0447\u0435\u0441\u043a\u0438\u0445 \u0434\u043e\u043a\u0443\u043c\u0435\u043d\u0442\u043e\u0432 \u043d\u0430 \u0441\u043b\u0435\u0434\u0443\u044e\u0449\u0438\u0435 100 \u043b\u0435\u0442, \u0441 \u0430\u043a\u0446\u0435\u043d

In [7]:
d02_total_dt

,result_name,result_value
0,comparison_winner,PDF
1,a_concept_score,11%
2,b_concept_score,-11%
3,LLM_model,deepseek/deepseek-pro
4,superordinate_concept,file format
5,perspective_observer_strategy,Archival institution perspective
6,point_of_view,Large organization (archive) selecting a file ...
7,observer_context_description,"Крупная организация (архив) должна выбрать, ка..."
8,a_concept,PDF
9,b_concept,TIFF


In [8]:
d02_aspect_dt

,type,aspect,feature,a_concept_winner,a_concept_score,b_concept_winner,b_concept_score
0,feature_score,Format stability and longevity,Standardization status,False,-,False,-
1,feature_score,Format stability and longevity,Adoption level in archival community,False,-,False,-
2,feature_score,Format stability and longevity,Backward compatibility track record,False,-,False,-
3,feature_score,Openness and independence from proprietary sof...,License type,False,-,False,-
4,feature_score,Openness and independence from proprietary sof...,Patent encumbrance,False,-,False,-
5,feature_score,Openness and independence from proprietary sof...,Availability of multiple independent implement...,False,-,False,-
6,feature_score,Self-documentation and metadata support,Internal metadata capacity,False,-,False,-
7,feature_score,Self-documentation and metadata support,Technical metadata support,False,-,False,-
8,feature_score,Self-documentation and metadata support,Fixity information support,False,-,False,-
9,feature_score,Robustness against corruption,Error detection mechanisms,False,-,False,-


## Пример №3: Выбор лучшей стратегии для снижения углеродных выбросов в крупном&nbsp;городе

In [ ]:
d03_observer_context_description = 'Мэрия города с населением 1 миллион человек должна выбрать, какой из двух проектов быстрее и эффективнее снизит выбросы CO2 на 15% в течение 5 лет.'
d03_a_concept = 'Масштабное субсидирование перехода на электромобили'
d03_b_concept = 'Модернизация и электрификация общественного транспорта'
d03_result = await calc_concept_aspect_comparison(d03_observer_context_description, d03_a_concept, d03_b_concept, frame_of_reference, output_content_language, extra_information_retrieval_strategy)
(d03_total_dt, d03_aspect_dt) = create_output_tables(d03_result, d03_observer_context_description, d03_a_concept, d03_b_concept)


start: input_id=superordinate_concept_identification_af.1.7c4b139220d16dcec6ab5e8be56c4348.0b3d2777d2008ccc87bdd4d1d42101f2
process:
{"concepts": "- \u041c\u0430\u0441\u0448\u0442\u0430\u0431\u043d\u043e\u0435 \u0441\u0443\u0431\u0441\u0438\u0434\u0438\u0440\u043e\u0432\u0430\u043d\u0438\u0435 \u043f\u0435\u0440\u0435\u0445\u043e\u0434\u0430 \u043d\u0430 \u044d\u043b\u0435\u043a\u0442\u0440\u043e\u043c\u043e\u0431\u0438\u043b\u0438\n- \u041c\u043e\u0434\u0435\u0440\u043d\u0438\u0437\u0430\u0446\u0438\u044f \u0438 \u044d\u043b\u0435\u043a\u0442\u0440\u0438\u0444\u0438\u043a\u0430\u0446\u0438\u044f \u043e\u0431\u0449\u0435\u0441\u0442\u0432\u0435\u043d\u043d\u043e\u0433\u043e \u0442\u0440\u0430\u043d\u0441\u043f\u043e\u0440\u0442\u0430", "context_knowledge_specification": "### Context description\n\u041c\u044d\u0440\u0438\u044f \u0433\u043e\u0440\u043e\u0434\u0430 \u0441 \u043d\u0430\u0441\u0435\u043b\u0435\u043d\u0438\u0435\u043c 1 \u043c\u0438\u043b\u043b\u0438\u043e\u043d \u0447\u0435

In [ ]:
d03_total_dt

,result_name,result_value
0,comparison_winner,Модернизация и электрификация общественного тр...
1,a_concept_score,-93%
2,b_concept_score,93%
3,LLM_model,deepseek/deepseek-chat
4,superordinate_concept,Меры по сокращению выбросов CO2 в транспортном...
5,perspective_observer_strategy,Стратегия принятия решений на основе сравнител...
6,point_of_view,"Точка зрения муниципального органа власти, отв..."
7,observer_context_description,Мэрия города с населением 1 миллион человек до...
8,a_concept,Масштабное субсидирование перехода на электром...
9,b_concept,Модернизация и электрификация общественного тр...


In [ ]:
d03_aspect_dt

,type,aspect,feature,a_concept_winner,a_concept_score,b_concept_winner,b_concept_score
0,feature_score,Скорость достижения целевого снижения выбросов,Время до достижения 15% снижения,False,-,True,12%
1,feature_score,Скорость достижения целевого снижения выбросов,Динамика снижения выбросов по годам,False,-,True,9%
2,feature_score,Эффективность мер (результат на единицу затрат),Снижение выбросов CO2 на единицу бюджетных затрат,False,-,True,8%
3,feature_score,Эффективность мер (результат на единицу затрат),Общие капитальные затраты,False,-,True,6%
4,feature_score,Эффективность мер (результат на единицу затрат),Эксплуатационные расходы,False,-,True,5%
5,feature_score,Масштаб воздействия и охват,"Доля транспортного сектора города, на которую ...",False,-,True,8%
6,feature_score,Масштаб воздействия и охват,Абсолютное снижение выбросов CO2,False,-,True,9%
7,feature_score,Технологическая готовность и реализуемость,Уровень технологической готовности (TRL),False,-,True,5%
8,feature_score,Технологическая готовность и реализуемость,Наличие необходимой инфраструктуры,False,-,True,4%
9,feature_score,Технологическая готовность и реализуемость,Сроки развертывания и внедрения,False,-,True,6%


## Пример №4: Выбор антисептика для бытовой обработки небольшой раны у ребёнка

In [ ]:
d04_observer_context_description = 'Выбор антисептика для бытовой обработки небольшой раны у ребёнка (дома, без аллергий): кухонный порез, нужно убить бактерии и не повредить заживление'
d04_a_concept = 'изопропиловый спирт (70%)'
d04_b_concept = 'раствор хлоргексидина'
d04_result = await calc_concept_aspect_comparison(d04_observer_context_description, d04_a_concept, d04_b_concept, frame_of_reference, output_content_language, extra_information_retrieval_strategy)
(d04_total_dt, d04_aspect_dt) = create_output_tables(d04_result, d04_observer_context_description, d04_a_concept, d04_b_concept)

start: input_id=superordinate_concept_identification_af.1.7c4b139220d16dcec6ab5e8be56c4348.6aee9b62acbe5bbc2b6a22ce3001d195
process:
{"concepts": "- \u0438\u0437\u043e\u043f\u0440\u043e\u043f\u0438\u043b\u043e\u0432\u044b\u0439 \u0441\u043f\u0438\u0440\u0442 (70%)\n- \u0440\u0430\u0441\u0442\u0432\u043e\u0440 \u0445\u043b\u043e\u0440\u0433\u0435\u043a\u0441\u0438\u0434\u0438\u043d\u0430", "context_knowledge_specification": "### Context description\n\u0412\u044b\u0431\u043e\u0440 \u0430\u043d\u0442\u0438\u0441\u0435\u043f\u0442\u0438\u043a\u0430 \u0434\u043b\u044f \u0431\u044b\u0442\u043e\u0432\u043e\u0439 \u043e\u0431\u0440\u0430\u0431\u043e\u0442\u043a\u0438 \u043d\u0435\u0431\u043e\u043b\u044c\u0448\u043e\u0439 \u0440\u0430\u043d\u044b \u0443 \u0440\u0435\u0431\u0451\u043d\u043a\u0430 (\u0434\u043e\u043c\u0430, \u0431\u0435\u0437 \u0430\u043b\u043b\u0435\u0440\u0433\u0438\u0439): \u043a\u0443\u0445\u043e\u043d\u043d\u044b\u0439 \u043f\u043e\u0440\u0435\u0437, \u043d\u0443\u0436\u043d

In [ ]:
d04_total_dt

,result_name,result_value
0,comparison_winner,раствор хлоргексидина
1,a_concept_score,-9%
2,b_concept_score,9%
3,LLM_model,deepseek/deepseek-chat
4,superordinate_concept,антисептик для обработки кожи
5,perspective_observer_strategy,Рациональный выбор на основе медицинских факто...
6,point_of_view,"Заботливый родитель, выбирающий средство для б..."
7,observer_context_description,Выбор антисептика для бытовой обработки неболь...
8,a_concept,изопропиловый спирт (70%)
9,b_concept,раствор хлоргексидина


In [ ]:
d04_aspect_dt

,type,aspect,feature,a_concept_winner,a_concept_score,b_concept_winner,b_concept_score
0,feature_score,Эффективность противомикробного действия,Спектр антимикробной активности,False,-,True,17%
1,feature_score,Эффективность противомикробного действия,Скорость наступления эффекта,True,13%,False,-
2,feature_score,Безопасность для детской кожи и заживления,Отсутствие негативного влияния на регенерацию ...,False,-,True,11%
3,feature_score,Безопасность для детской кожи и заживления,Низкий риск раздражения,False,-,True,10%
4,feature_score,Безопасность для детской кожи и заживления,Отсутствие системной токсичности при местном п...,False,-,True,10%
5,feature_score,Удобство применения в быту,Форма выпуска,False,-,False,-
6,feature_score,Удобство применения в быту,Простота нанесения,False,-,False,-
7,feature_score,Удобство применения в быту,Отсутствие сильного окрашивания кожи,True,6%,False,-
8,feature_score,Доступность и экономичность,Цена,True,10%,False,-
9,feature_score,Доступность и экономичность,Широта распространения в аптеках,True,8%,False,-


## Пример №5: Выбор смартфона для фрилансера-дизайнера

In [ ]:
d05_observer_context_description = 'Работаю фрилансером-дизайнером, много времени провожу в Adobe Photoshop и Lightroom на мобильном, часто редактирую фото/видео на ходу, синхронизирую с MacBook, важна стабильность ПО и автономность батареи (день без подзарядки)'
d05_a_concept = 'Google Pixel 9 Pro'
d05_b_concept = 'iPhone 14 Pro'
d05_result = await calc_concept_aspect_comparison(d05_observer_context_description, d05_a_concept, d05_b_concept, frame_of_reference, output_content_language, extra_information_retrieval_strategy)
(d05_total_dt, d05_aspect_dt) = create_output_tables(d05_result, d05_observer_context_description, d05_a_concept, d05_b_concept)

start: input_id=superordinate_concept_identification_af.1.7c4b139220d16dcec6ab5e8be56c4348.568c87a3b31c9a2635b3337bc799fc1c
process:
{"concepts": "- Google Pixel 9 Pro\n- iPhone 14 Pro", "context_knowledge_specification": "### Context description\n\u0420\u0430\u0431\u043e\u0442\u0430\u044e \u0444\u0440\u0438\u043b\u0430\u043d\u0441\u0435\u0440\u043e\u043c-\u0434\u0438\u0437\u0430\u0439\u043d\u0435\u0440\u043e\u043c, \u043c\u043d\u043e\u0433\u043e \u0432\u0440\u0435\u043c\u0435\u043d\u0438 \u043f\u0440\u043e\u0432\u043e\u0436\u0443 \u0432 Adobe Photoshop \u0438 Lightroom \u043d\u0430 \u043c\u043e\u0431\u0438\u043b\u044c\u043d\u043e\u043c, \u0447\u0430\u0441\u0442\u043e \u0440\u0435\u0434\u0430\u043a\u0442\u0438\u0440\u0443\u044e \u0444\u043e\u0442\u043e/\u0432\u0438\u0434\u0435\u043e \u043d\u0430 \u0445\u043e\u0434\u0443, \u0441\u0438\u043d\u0445\u0440\u043e\u043d\u0438\u0437\u0438\u0440\u0443\u044e \u0441 MacBook, \u0432\u0430\u0436\u043d\u0430 \u0441\u0442\u0430\u0431\u0438\u043b\u044

In [ ]:
d05_total_dt

,result_name,result_value
0,comparison_winner,iPhone 14 Pro
1,a_concept_score,-21%
2,b_concept_score,21%
3,LLM_model,deepseek/deepseek-chat
4,superordinate_concept,flagship smartphones
5,perspective_observer_strategy,Прагматичный профессионал
6,point_of_view,"Мобильный дизайнер-фрилансер, зависящий от про..."
7,observer_context_description,"Работаю фрилансером-дизайнером, много времени ..."
8,a_concept,Google Pixel 9 Pro
9,b_concept,iPhone 14 Pro


In [ ]:
d05_aspect_dt

,type,aspect,feature,a_concept_winner,a_concept_score,b_concept_winner,b_concept_score
0,feature_score,Производительность и стабильность системы,Процессор,False,-,True,8%
1,feature_score,Производительность и стабильность системы,Оперативная память (ОЗУ),True,7%,False,-
2,feature_score,Производительность и стабильность системы,Стабильность операционной системы,False,-,True,9%
3,feature_score,Автономность,Емкость аккумулятора,True,7%,False,-
4,feature_score,Автономность,Время автономной работы при активном использов...,False,-,True,9%
5,feature_score,Автономность,Поддержка быстрой зарядки,True,6%,False,-
6,feature_score,Экран,Разрешение экрана,True,8%,False,-
7,feature_score,Экран,Частота обновления экрана,False,-,False,-
8,feature_score,Экран,Яркость,False,-,True,6%
9,feature_score,Интеграция с экосистемой,Беспрепятственная синхронизация с ПК/Mac,False,-,True,10%


## Пример №6: Выбор языковой модели для типовых задач в области компьютерных наук

In [ ]:
d06_observer_context_description = 'Типовые рабочие задачи в области компьтерных наук и исследований'
d06_a_concept = 'ChatGPT (LLM)'
d06_b_concept = 'Claude Opus (LLM)'
d06_result = await calc_concept_aspect_comparison(d06_observer_context_description, d06_a_concept, d06_b_concept, frame_of_reference, output_content_language, extra_information_retrieval_strategy)
(d06_total_dt, d06_aspect_dt) = create_output_tables(d06_result, d06_observer_context_description, d06_a_concept, d06_b_concept)

start: input_id=superordinate_concept_identification_af.1.3f8914212a77e71bfe150190c5b8514c.5d6b6c800e157925a9d59df4b60f7aa7
process:
{"concepts": "- ChatGPT (LLM)\n- Claude Opus (LLM)", "context_knowledge_specification": "### Context description\n\u0422\u0438\u043f\u043e\u0432\u044b\u0435 \u0440\u0430\u0431\u043e\u0447\u0438\u0435 \u0437\u0430\u0434\u0430\u0447\u0438 \u0432 \u043e\u0431\u043b\u0430\u0441\u0442\u0438 \u043a\u043e\u043c\u043f\u044c\u0442\u0435\u0440\u043d\u044b\u0445 \u043d\u0430\u0443\u043a \u0438 \u0438\u0441\u0441\u043b\u0435\u0434\u043e\u0432\u0430\u043d\u0438\u0439", "meta": {"model": "deepseek/deepseek-pro"}, "process_type": "superordinate_concept_identification_af", "input_id": "superordinate_concept_identification_af.1.3f8914212a77e71bfe150190c5b8514c.5d6b6c800e157925a9d59df4b60f7aa7"}
end: input_id=superordinate_concept_identification_af.1.3f8914212a77e71bfe150190c5b8514c.5d6b6c800e157925a9d59df4b60f7aa7
other_notes: Both ChatGPT and Claude Opus are specific ins

In [ ]:
d06_total_dt

,result_name,result_value
0,comparison_winner,Claude Opus (LLM)
1,a_concept_score,-83%
2,b_concept_score,83%
3,LLM_model,deepseek/deepseek-pro
4,superordinate_concept,Large Language Model
5,perspective_observer_strategy,Objective analysis of typical work tasks in co...
6,point_of_view,Researcher/Computer Scientist
7,observer_context_description,Типовые рабочие задачи в области компьтерных н...
8,a_concept,ChatGPT (LLM)
9,b_concept,Claude Opus (LLM)


In [ ]:
d06_aspect_dt

,type,aspect,feature,a_concept_winner,a_concept_score,b_concept_winner,b_concept_score
0,feature_score,Функциональные возможности,Генерация текста,False,-,True,4%
1,feature_score,Функциональные возможности,Ответы на вопросы,False,-,True,4%
2,feature_score,Функциональные возможности,Перевод,False,-,True,3%
3,feature_score,Функциональные возможности,Суммаризация,False,-,True,3%
4,feature_score,Функциональные возможности,Генерация кода,False,-,True,4%
5,feature_score,Функциональные возможности,Анализ данных,False,-,True,3%
6,feature_score,Производительность,Скорость инференса,False,-,True,7%
7,feature_score,Производительность,Пропускная способность,False,-,True,7%
8,feature_score,Производительность,Использование памяти,False,-,True,6%
9,feature_score,Точность и качество,Точность на бенчмарках,False,-,True,8%


In [ ]:
d07_observer_context_description = 'Действует нешаблонно и проактивно при одинаковом качестве ответа'
d07_a_concept = 'ChatGPT (LLM)'
d07_b_concept = 'Claude Opus (LLM)'
d07_result = await calc_concept_aspect_comparison(d07_observer_context_description, d07_a_concept, d07_b_concept, frame_of_reference, output_content_language, extra_information_retrieval_strategy)
(d07_total_dt, d07_aspect_dt) = create_output_tables(d07_result, d07_observer_context_description, d07_a_concept, d07_b_concept)

start: input_id=superordinate_concept_identification_af.1.3f8914212a77e71bfe150190c5b8514c.28b44b5b87f2093ad38a56e29c3ccb41
process:
{"concepts": "- ChatGPT (LLM)\n- Claude Opus (LLM)", "context_knowledge_specification": "### Context description\n\u0414\u0435\u0439\u0441\u0442\u0432\u0443\u0435\u0442 \u043d\u0435\u0448\u0430\u0431\u043b\u043e\u043d\u043d\u043e \u0438 \u043f\u0440\u043e\u0430\u043a\u0442\u0438\u0432\u043d\u043e \u043f\u0440\u0438 \u043e\u0434\u0438\u043d\u0430\u043a\u043e\u0432\u043e\u043c \u043a\u0430\u0447\u0435\u0441\u0442\u0432\u0435 \u043e\u0442\u0432\u0435\u0442\u0430", "meta": {"model": "deepseek/deepseek-pro"}, "process_type": "superordinate_concept_identification_af", "input_id": "superordinate_concept_identification_af.1.3f8914212a77e71bfe150190c5b8514c.28b44b5b87f2093ad38a56e29c3ccb41"}
end: input_id=superordinate_concept_identification_af.1.3f8914212a77e71bfe150190c5b8514c.28b44b5b87f2093ad38a56e29c3ccb41
other_notes: Both ChatGPT and Claude Opus are specifi

In [ ]:
d07_total_dt

,result_name,result_value
0,comparison_winner,Claude Opus (LLM)
1,a_concept_score,-16%
2,b_concept_score,16%
3,LLM_model,deepseek/deepseek-pro
4,superordinate_concept,Large Language Model
5,perspective_observer_strategy,Proactive non-template behavior with consisten...
6,point_of_view,Observer evaluating LLM behavior
7,observer_context_description,Действует нешаблонно и проактивно при одинаков...
8,a_concept,ChatGPT (LLM)
9,b_concept,Claude Opus (LLM)


In [ ]:
d07_aspect_dt

,type,aspect,feature,a_concept_winner,a_concept_score,b_concept_winner,b_concept_score
0,feature_score,Нешаблонность,Степень отклонения от шаблона,False,-,True,16%
1,feature_score,Нешаблонность,Частота нешаблонных ответов,False,-,False,-
2,feature_score,Проактивность,Инициативность,False,-,False,-
3,feature_score,Проактивность,Предугадывание потребностей,False,-,False,-
4,feature_score,Качество ответа,Точность,False,-,False,-
5,feature_score,Качество ответа,Полнота,False,-,False,-
6,feature_score,Качество ответа,Релевантность,False,-,False,-


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
#!cp  -r /content/processes /content/drive/MyDrive/processes